In [1]:
import csv
from itertools import combinations
from collections import defaultdict, Counter

class FPTreeNode:
    """Узел FP-дерева"""
    def __init__(self, item, count, parent):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None  # ссылка на следующий узел с таким же товаром

    def increment(self, count):
        """Увеличивает счетчик узла"""
        self.count += count


def build_fp_tree(transactions, min_support_count):
    """
    Строит FP-дерево из транзакций
    
    transactions: список транзакций
    min_support_count: минимальное абсолютное значение поддержки
    """
    # Шаг 1: Считаем поддержку для отдельных товаров
    item_counts = Counter()
    for transaction in transactions:
        for item in transaction:
            item_counts[item] += 1
    
    # Фильтруем по минимальной поддержке и сортируем
    frequent_items = {item: count for item, count in item_counts.items() 
                     if count >= min_support_count}
    
    if not frequent_items:
        return None, None, None
    
    # Сортируем по убыванию поддержки
    sorted_frequent_items = sorted(frequent_items.items(), 
                                  key=lambda x: (-x[1], x[0]))
    
    # Таблица заголовков
    header_table = {item: [count, None] for item, count in sorted_frequent_items}
    
    # Создаем корневой узел
    root = FPTreeNode(None, 1, None)
    
    # Шаг 2: Строим дерево
    for transaction in transactions:
        # Фильтруем и сортируем товары в транзакции
        filtered_items = [item for item in transaction if item in frequent_items]
        if not filtered_items:
            continue
            
        # Сортируем по убыванию поддержки
        filtered_items.sort(key=lambda x: (-frequent_items[x], x))
        
        # Вставляем в дерево
        current_node = root
        for item in filtered_items:
            if item in current_node.children:
                child = current_node.children[item]
                child.increment(1)
            else:
                # Создаем новый узел
                child = FPTreeNode(item, 1, current_node)
                current_node.children[item] = child
                
                # Обновляем таблицу заголовков
                if header_table[item][1] is None:
                    header_table[item][1] = child
                else:
                    # Добавляем в связный список
                    node = header_table[item][1]
                    while node.link is not None:
                        node = node.link
                    node.link = child
            
            current_node = child
    
    return root, header_table, frequent_items


def find_prefix_path(base_item, header_node):
    """
    Находит все пути от заданного товара до корня
    """
    paths = []
    node = header_node
    
    while node is not None:
        # Находим путь к корню
        path = []
        current = node.parent
        while current.item is not None:
            path.append(current.item)
            current = current.parent
        
        if path:
            paths.append((path, node.count))
        
        node = node.link
    
    return paths


def mine_fp_tree(header_table, min_support_count, prefix, frequent_itemsets, item_counts, k=0):
    """
    Рекурсивно добывает частые наборы из FP-дерева
    
    header_table: таблица заголовков
    min_support_count: минимальная поддержка
    prefix: текущий префикс (набор товаров)
    frequent_itemsets: словарь для сохранения результатов
    item_counts: счетчики для поддержки
    k: текущий размер набора
    """
    # Сортируем товары по поддержке
    sorted_items = sorted(header_table.items(), 
                         key=lambda x: x[1][0])  # сортируем по поддержке
    
    for item, (count, node) in sorted_items:
        # Создаем новый набор
        new_prefix = prefix.copy()
        new_prefix.append(item)
        new_prefix_tuple = tuple(sorted(new_prefix))
        
        # Сохраняем частый набор
        if k not in frequent_itemsets:
            frequent_itemsets[k] = {}
        frequent_itemsets[k][new_prefix_tuple] = count / item_counts['total_transactions']
        
        # Строим условное дерево
        conditional_paths = find_prefix_path(item, node)
        conditional_transactions = []
        
        for path, path_count in conditional_paths:
            # Добавляем путь нужное количество раз
            for _ in range(path_count):
                conditional_transactions.append(path)
        
        # Строим условное FP-дерево
        if conditional_transactions:
            conditional_root, conditional_header, _ = build_fp_tree(
                conditional_transactions, min_support_count)
            
            if conditional_header is not None:
                # Рекурсивно добываем частые наборы
                mine_fp_tree(conditional_header, min_support_count, 
                            new_prefix, frequent_itemsets, item_counts, k + 1)


def fp_growth(transactions, min_support=0.5):
    """
    Реализация алгоритма FP-Growth
    
    transactions: список транзакций (каждая транзакция - список товаров)
    min_support: минимальная поддержка (например, 0.5 = 50%)
    """
    num_transactions = len(transactions)
    min_support_count = int(min_support * num_transactions)
    
    # Сохраняем общее количество транзакций
    item_counts = {'total_transactions': num_transactions}
    
    # Строим FP-дерево
    root, header_table, frequent_items = build_fp_tree(transactions, min_support_count)
    
    if root is None:
        return {1: {}}
    
    # Добываем частые наборы
    frequent_itemsets = {}
    prefix = []
    
    # Сначала добавляем одиночные частые товары
    frequent_itemsets[1] = {}
    for item, count in sorted(frequent_items.items()):
        support = count / num_transactions
        frequent_itemsets[1][(item,)] = support
    
    # Добываем наборы большего размера
    mine_fp_tree(header_table, min_support_count, prefix, 
                frequent_itemsets, item_counts, 1)
    
    # Удаляем пустые уровни и сортируем
    result = {}
    for k in sorted(frequent_itemsets.keys()):
        if frequent_itemsets[k]:
            # Сортируем по поддержке
            sorted_items = sorted(frequent_itemsets[k].items(), 
                                 key=lambda x: (-x[1], x[0]))
            result[k] = dict(sorted_items)
            
            print(f"\nЧастые наборы из {k} товаров:")
            for itemset, support in result[k].items():
                print(f"  {set(itemset)} - поддержка: {support:.2%}")
    
    return result


def generate_candidates(frequent_k, k):
    """Генерирует кандидатов размера k+1 из частых наборов размера k"""
    candidates = set()
    n = len(frequent_k)
    frequent_k_set = set(frequent_k) if k > 1 else None
    
    for i in range(n):
        for j in range(i + 1, n):
            # Объединяем два набора
            itemset1 = frequent_k[i]
            itemset2 = frequent_k[j]
            
            if k == 1:
                # Для одиночных элементов просто объединяем
                new_candidate = tuple(sorted([itemset1[0], itemset2[0]]))
                candidates.add(new_candidate)
            else:
                # Для k > 1: проверяем, что первые k-1 элементов совпадают
                if itemset1[:-1] == itemset2[:-1]:
                    # Объединяем последние элементы
                    new_candidate = tuple(sorted(set(itemset1) | set(itemset2)))
                    
                    # Проверяем, что все (k-1)-подмножества частые (apriori property)
                    if has_infrequent_subset(new_candidate, frequent_k_set, k):
                        continue
                    
                    candidates.add(new_candidate)
    
    return sorted(candidates)


def has_infrequent_subset(candidate, frequent_k_set, subset_size):
    """Проверяет, есть ли у кандидата нечастые подмножества размера subset_size"""
    for subset in combinations(candidate, subset_size):
        subset = tuple(sorted(subset))
        if subset not in frequent_k_set:
            return True
    return False


def calculate_support(itemset, transactions):
    """Считает поддержку для набора товаров"""
    count = 0
    itemset_set = set(itemset)
    for transaction in transactions:
        if itemset_set.issubset(set(transaction)):
            count += 1
    return count / len(transactions)


def generate_rules(frequent_items, transactions, min_confidence=0.7):
    """
    Генерирует правила ассоциации по частым наборам.

    frequent_items: словарь вида
        k -> { itemset (кортеж товаров) : support (поддержка) }
    min_confidence: минимально допустимая достоверность правила
    """
    rules = []

    # Подготовим быстрый доступ к поддержкам для любых наборов
    # (чтобы не пересчитывать поддержку каждый раз с нуля)
    support_lookup = {}
    for size_dict in frequent_items.values():
        support_lookup.update(size_dict)

    for k in frequent_items:
        if k < 2:  # Для правил нужны наборы из 2+ товаров
            continue

        for itemset, support_full in frequent_items[k].items():
            itemset_set = set(itemset)

            # Генерируем все возможные правила
            for i in range(1, k):
                # Все комбинации для антецедента (левой части правила)
                for antecedent in combinations(itemset, i):
                    antecedent_set = set(antecedent)
                    consequent_set = itemset_set - antecedent_set

                    # Берём поддержку антецедента (если нет в словаре — считаем)
                    antecedent_tuple = tuple(sorted(antecedent_set))
                    support_antecedent = support_lookup.get(
                        antecedent_tuple,
                        calculate_support(antecedent_tuple, transactions)
                    )
                    confidence = support_full / support_antecedent if support_antecedent > 0 else 0

                    if confidence >= min_confidence:
                        # Берём поддержку консеквента (правой части правила)
                        consequent_tuple = tuple(sorted(consequent_set))
                        support_consequent = support_lookup.get(
                            consequent_tuple,
                            calculate_support(consequent_tuple, transactions)
                        )

                        # Lift = доверие(A -> B) / support(B)
                        lift = confidence / support_consequent if support_consequent > 0 else 0

                        rules.append({
                            'antecedent': antecedent_set,
                            'consequent': consequent_set,
                            'support': support_full,
                            'confidence': confidence,
                            'lift': lift
                        })

    return rules


def read_transactions_from_csv(file_path):
    """
    Читает транзакции из CSV файла.
    Предполагается, что каждая строка - это транзакция, товары разделены запятыми.
    Без заголовков.
    """
    transactions = []
    with open(file_path, 'r', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            # Удаляем пустые строки и очищаем товары от лишних пробелов
            cleaned_row = [item.strip() for item in row if item.strip()]
            if cleaned_row:
                transactions.append(cleaned_row)
    return transactions


# Тестовый пример
def test_fp_growth(csv_file_path):
    # Читаем транзакции из CSV
    transactions = read_transactions_from_csv(csv_file_path)

    print("Транзакции:")
    for i, t in enumerate(transactions, 1):
        print(f"{i}: {t}")

    print("\n" + "=" * 50)
    print("ЗАПУСК FP-GROWTH")
    print("=" * 50)

    # Запускаем FP-Growth
    min_support = 0.4  # 40%
    frequent_items = fp_growth(transactions, min_support)

    print("\n" + "=" * 50)
    print("ГЕНЕРАЦИЯ ПРАВИЛ")
    print("=" * 50)

    # Генерируем правила
    min_confidence = 0.6  # 60%
    rules = generate_rules(frequent_items, transactions, min_confidence)

    print(f"\nНайдено правил: {len(rules)}")
    for i, rule in enumerate(rules, 1):
        print(f"\nПравило {i}:")
        print(f"  {rule['antecedent']} → {rule['consequent']}")
        print(f"  Поддержка: {rule['support']:.2%}")
        print(f"  Достоверность: {rule['confidence']:.2%}")
        print(f"  Lift: {rule['lift']:.2f}")


# Пример использования
if __name__ == "__main__":
    test_fp_growth(r'C:\Users\Пользователь\Desktop\програмироание\mecanicoratrismeckanikkavenal\lab-4\apriori_db_1.csv')


# Поддержка (support) – доля всех транзакций, где одновременно есть и левая, и правая часть правила.
# Достоверность (confidence) – среди транзакций, где есть левая часть (A), в какой доле также есть правая (B).
# Lift – во сколько раз правило A -> B сильнее (или слабее) простой совместной встречаемости B.

Транзакции:
1: ['хлеб', 'вода', 'молоко']
2: ['масло', 'вода', 'орехи']
3: ['шоколад', 'хлеб', 'масло', 'вода', 'орехи']
4: ['масло', 'молоко', 'орехи']

ЗАПУСК FP-GROWTH

Частые наборы из 1 товаров:
  {'вода'} - поддержка: 75.00%
  {'масло'} - поддержка: 75.00%
  {'орехи'} - поддержка: 75.00%
  {'молоко'} - поддержка: 50.00%
  {'хлеб'} - поддержка: 50.00%
  {'шоколад'} - поддержка: 25.00%

Частые наборы из 2 товаров:
  {'масло', 'орехи'} - поддержка: 75.00%
  {'вода', 'масло'} - поддержка: 50.00%
  {'вода', 'орехи'} - поддержка: 50.00%
  {'хлеб', 'вода'} - поддержка: 50.00%
  {'вода', 'молоко'} - поддержка: 25.00%
  {'вода', 'шоколад'} - поддержка: 25.00%
  {'масло', 'молоко'} - поддержка: 25.00%
  {'хлеб', 'масло'} - поддержка: 25.00%
  {'шоколад', 'масло'} - поддержка: 25.00%
  {'орехи', 'молоко'} - поддержка: 25.00%
  {'хлеб', 'молоко'} - поддержка: 25.00%
  {'хлеб', 'орехи'} - поддержка: 25.00%
  {'шоколад', 'орехи'} - поддержка: 25.00%
  {'хлеб', 'шоколад'} - поддержка: 25.00%

Ч